## 1. Instalação de dependências

Execute esta célula apenas se alguma biblioteca estiver faltando no seu ambiente.

In [1]:
# Execute se necessário:
# %pip install -U pandas numpy matplotlib scikit-learn tensorflow ydata-profiling setuptools

## 2. Importações

In [6]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ydata_profiling é opcional. Se não estiver instalado, o restante do notebook continua rodando.
try:
    from ydata_profiling import ProfileReport
    YDATA_AVAILABLE = True
except Exception as e:
    YDATA_AVAILABLE = False
    print("ydata_profiling não está disponível. O relatório automático será pulado.")
    print("Motivo:", e)

# TensorFlow/Keras é usado para a rede neural.
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Input
    TF_AVAILABLE = True
except Exception as e:
    TF_AVAILABLE = False
    print("TensorFlow não está disponível neste ambiente.")
    print("Execute a célula de instalação acima ou instale tensorflow no seu ambiente.")
    print("Motivo:", e)

ydata_profiling não está disponível. O relatório automático será pulado.
Motivo: No module named 'pkg_resources'


## 3. Carregamento dos dados

O arquivo `olist_orders_dataset.csv` precisa estar na mesma pasta do notebook. Caso esteja em outra pasta, ajuste a variável `DATA_PATH`.

In [7]:
DATA_PATH = Path("olist_orders_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH.resolve()}
"
        "Coloque o arquivo olist_orders_dataset.csv na mesma pasta deste notebook "
        "ou altere a variável DATA_PATH para o caminho correto."
    )

df = pd.read_csv(DATA_PATH, sep=",", on_bad_lines="skip")

print("Dimensão do dataset:", df.shape)
df.head()

SyntaxError: unterminated f-string literal (detected at line 5) (241341837.py, line 5)

## 4. Análise exploratória inicial

In [ ]:
df.describe(include="all").T

In [ ]:
df.info()

In [ ]:
if YDATA_AVAILABLE:
    profile = ProfileReport(df, title="Profiling Report", minimal=True)
    profile.to_notebook_iframe()
else:
    print("Relatório automático pulado porque ydata_profiling não está instalado/disponível.")

## 5. Pré-processamento

In [ ]:
cols_datas = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Converte datas com segurança. Valores inválidos viram NaT em vez de quebrar o notebook.
for col in cols_datas:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Mantém apenas pedidos entregues e com previsão de entrega, pois o alvo depende dessas datas.
df_model = df.dropna(subset=[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]).copy()

print("Linhas após remover datas essenciais ausentes:", df_model.shape[0])

In [ ]:
# Target: 1 = entrega atrasada, 0 = entrega no prazo
df_model["atraso"] = (
    df_model["order_delivered_customer_date"] > df_model["order_estimated_delivery_date"]
).astype(int)

df_model["atraso"].value_counts(normalize=True).rename("proporcao")

In [ ]:
# Features derivadas de datas disponíveis no dataset
df_model["tempo_entrega"] = (
    df_model["order_delivered_customer_date"] - df_model["order_purchase_timestamp"]
).dt.days

df_model["tempo_estimado"] = (
    df_model["order_estimated_delivery_date"] - df_model["order_purchase_timestamp"]
).dt.days

df_model["diff_estimado_real"] = df_model["tempo_entrega"] - df_model["tempo_estimado"]

features = ["tempo_entrega", "tempo_estimado", "diff_estimado_real"]

df_model = df_model.dropna(subset=features + ["atraso"]).copy()

print("Linhas finais para modelagem:", df_model.shape[0])
df_model[features + ["atraso"]].head()

> Observação: `tempo_entrega` e `diff_estimado_real` usam a data real de entrega. Isso faz sentido para análise do atraso, mas para prever antes da entrega pode causar vazamento de informação. Para previsão real antecipada, o ideal seria usar variáveis conhecidas no momento da compra, como estado do cliente, estado do vendedor, frete, categoria, preço etc.

## 6. Separação entre treino e teste

In [ ]:
X = df_model[features]
y = df_model["atraso"]

stratify_param = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_param
)

# Ajusta o scaler somente no treino para evitar vazamento de dados.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Treino:", X_train_scaled.shape)
print("Teste:", X_test_scaled.shape)

## 7. Modelo de Rede Neural — ReLU

In [ ]:
if not TF_AVAILABLE:
    raise ImportError(
        "TensorFlow não está disponível. Execute a célula de instalação no início do notebook "
        "e reinicie o kernel antes de continuar."
    )

model_relu = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_relu.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_relu.summary()

In [ ]:
history_relu = model_relu.fit(
    X_train_scaled,
    y_train,
    epochs=6,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    verbose=1
)

## 8. Gráfico de treinamento — ReLU

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_relu.history["loss"], label="Loss treino")
plt.plot(history_relu.history["val_loss"], label="Loss validação")
plt.plot(history_relu.history["accuracy"], "--", label="Accuracy treino")
plt.plot(history_relu.history["val_accuracy"], "--", label="Accuracy validação")
plt.title("Comportamento do modelo ReLU durante o treinamento")
plt.xlabel("Épocas")
plt.ylabel("Métricas")
plt.legend()
plt.grid(True)
plt.show()

## 9. Avaliação do modelo — ReLU

In [ ]:
loss_relu, acc_relu = model_relu.evaluate(X_test_scaled, y_test, verbose=0)
print("Loss ReLU:", loss_relu)
print("Acurácia ReLU:", acc_relu)

In [ ]:
train_loss = history_relu.history["loss"]
val_loss = history_relu.history["val_loss"]

diff = np.mean(np.array(val_loss) - np.array(train_loss))

print("Diferença média (val_loss - loss):", diff)

if diff > 0.05:
    print("Indício de overfitting")
else:
    print("Sem overfitting significativo")

## 10. Modelo de Rede Neural — Sigmoid

In [ ]:
model_sigmoid = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(16, activation="sigmoid"),
    Dense(8, activation="sigmoid"),
    Dense(1, activation="sigmoid")
])

model_sigmoid.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_sigmoid = model_sigmoid.fit(
    X_train_scaled,
    y_train,
    epochs=7,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    verbose=1
)

loss_sigmoid, acc_sigmoid = model_sigmoid.evaluate(X_test_scaled, y_test, verbose=0)
print("Loss Sigmoid:", loss_sigmoid)
print("Acurácia Sigmoid:", acc_sigmoid)

## 11. Comparação ReLU vs Sigmoid

In [ ]:
comparacao_nn = pd.DataFrame({
    "Modelo": ["Rede Neural ReLU", "Rede Neural Sigmoid"],
    "Loss final validação": [history_relu.history["val_loss"][-1], history_sigmoid.history["val_loss"][-1]],
    "Acurácia teste": [acc_relu, acc_sigmoid]
})

comparacao_nn

## 12. Comparação com Regressão Logística

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

acc_lr = accuracy_score(y_test, y_pred_lr)
print("Acurácia Regressão Logística:", acc_lr)
print("
Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_lr))
print("
Relatório de classificação:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
comparacao_final = pd.DataFrame({
    "Modelo": ["Rede Neural ReLU", "Rede Neural Sigmoid", "Regressão Logística"],
    "Acurácia": [acc_relu, acc_sigmoid, acc_lr]
}).sort_values("Acurácia", ascending=False)

comparacao_final

## 13. Conclusão

1. **O modelo apresentou overfitting?**  
   A verificação foi feita comparando a diferença média entre `val_loss` e `loss`. Se a perda de validação ficar muito maior que a perda de treino, há indício de overfitting.

2. **Qual ativação foi melhor: ReLU ou Sigmoid?**  
   A comparação deve ser feita pela acurácia de teste e pelo `val_loss`. Em geral, ReLU tende a treinar melhor em camadas ocultas, mas o resultado final depende dos dados.

3. **A regressão logística foi competitiva?**  
   Como as features usadas são muito diretamente relacionadas ao atraso, a regressão logística pode apresentar desempenho muito alto. Para uma previsão real antes da entrega, seria necessário usar apenas dados disponíveis antes da entrega acontecer.